In [ ]:
#| default_exp machine_learning.model_registry

> **AI-generated — pending human review.** Created by Claude Code on 2026-10-21. Review checklist: `_nbdev_development/AI_REVIEW_CHECKLIST.md` (workspace root). Remove this cell only after review passes.

Model ids used to be string literals scattered across notebooks, tutorials, the GUI, and the playgrounds — which is how two generations of the same task coexisted (the GUI loading `def_and_notat_token_classification_model` while the library notebooks loaded `..._modernbert_base`).

This module is the **single source of truth**: consumers ask for a model by *task*, never by repo id. Replacing a model (consolidation, retraining, deprecation) is a one-line edit here, and superseded artifacts stay listed under `LEGACY_MODEL_IDS` so rollback is always one call away.

In [ ]:
#| export
MODEL_IDS = {
    'note_type': 'hyunjongkimmath/information_note_type',
    'def_notat_ner': 'hyunjongkimmath/def_and_notat_token_classification_model_modernbert_base',
    'notation_summarization': 'hyunjongkimmath/notation_summarizations_model',
    'notation_summarization_with_context': 'hyunjongkimmath/notation_summarizations_with_context_model',
    'definition_naming': 'hyunjongkimmath/definition_naming_model',
    'notation_naming': 'hyunjongkimmath/notation_naming_model',
    'note_linking': 'hyunjongkimmath/note_linking_classifier_modernbert',
    'notation_linking_rag': 'hyunjongkimmath/notation_linking_rag_modernbert',
    'notation_linking_embeddings': 'hyunjongkimmath/notation_linking_rag_sentence_transformers_all_MiniLM_L6_v2',
}

In [ ]:
#| export
LEGACY_MODEL_IDS = {
    'def_notat_ner': 'hyunjongkimmath/def_and_notat_token_classification_model',
}

`EMBEDDING_MODELS` records non-Hugging-Face embedders (served by Ollama) so a future swap is likewise a one-line edit; the index built by `machine_learning.semantic_search` should be rebuilt whenever the value changes.

In [ ]:
#| export
EMBEDDING_MODELS = {
    'semantic_search': 'nomic-embed-text',
}

In [ ]:
#| export
def get_model_id(task: str, legacy: bool = False) -> str:
    """
    Return the Hugging Face repo id serving `task`.

    With `legacy=True`, return the superseded artifact instead — kept
    working for rollback and for comparing generations on frozen evals.

    **Parameters**
    - `task` (*str*) — A key of `MODEL_IDS` (e.g. `'def_notat_ner'`).
    - `legacy` (*bool*) — Return the superseded artifact instead.
    """
    registry = LEGACY_MODEL_IDS if legacy else MODEL_IDS
    if task not in registry:
        raise KeyError(f'Unknown ML task {task!r}; known tasks: {sorted(MODEL_IDS)}.')
    return registry[task]

In [ ]:
from fastcore.test import *

In [ ]:
# Every registered artifact is a Hugging Face repo under the project org.
assert all(v.startswith('hyunjongkimmath/') for v in MODEL_IDS.values())

# The canonical NER model is the modernbert-base generation, not the legacy one.
test_eq(get_model_id('def_notat_ner'),
        'hyunjongkimmath/def_and_notat_token_classification_model_modernbert_base')
test_eq(get_model_id('def_notat_ner', legacy=True),
        'hyunjongkimmath/def_and_notat_token_classification_model')

# Exact pins for ids formerly hardcoded in the GUI (B2 flip): a registry edit
# that changes either value is an intentional cutover, and this test fires.
test_eq(get_model_id('note_type'), 'hyunjongkimmath/information_note_type')

# Legacy entries must name a task that also has a current artifact.
assert set(LEGACY_MODEL_IDS) <= set(MODEL_IDS)
test_ne(get_model_id('def_notat_ner'), get_model_id('def_notat_ner', legacy=True))

# Unknown tasks fail loudly with the list of known ones.
ExceptionExpected(lambda: get_model_id(task='bogus_task'))
